#  SYNTH-REALITY Engine: Signal Visualization Dashboard

This notebook orchestrates the full pipeline and renders a multi-panel diagnostic view:
1. **IPDA Memory Bands** (`L20, L40, L60`)
2. **Potential Attraction Field** (`$\Phi_{\text{mag}}$`)
3. **Gamma Feedback State** (Linear → Overclock → Crash)
4. **Spectral Misalignment** (`$M$` & Sync Score)
5. **Composite Signal Output** with regime tagging

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import sys; sys.path.append('../')

from core.ipda_window import IPDAWindow
from core.phi_magnet import PhiMagnet
from core.psi_sd_classifier import PSDClassifier
from core.gamma_amplifier import GammaAmplifier
from core.misalign_spectral import MisalignSpectral

plt.style.use('dark_background')

In [ ]:
# 🧬 Synthetic Market Generator
np.random.seed(42)
bars = 300
t = np.linspace(0, 20, bars)
trend = np.where(t > 8, 0.15 * (t - 8), 0.0)
cycles = np.sin(0.3 * t) * 2 + np.sin(0.08 * t) * 5
noise = np.random.normal(0, 0.4, bars)
prices = 100 + trend + cycles + noise
volume = np.exp(np.sin(0.2 * t)) * 500 + np.random.poisson(150, bars)

timestamps = pd.date_range('2024-01-01', periods=bars, freq='h')

In [ ]:
%%time
# ⚙️ Pipeline Execution
ipda = IPDAWindow(windows=(20, 40, 60), decay_rate=0.12)
magnet = PhiMagnet(gravity=0.75, exponent=1.8)
classifier = PSDClassifier(sweep_threshold=1.4, efficiency_floor=0.38)
gamma = GammaAmplifier(beta=0.45, crash_threshold=3.0, smoothing=4)
spectral = MisalignSpectral(fft_window=64, overlap=0.6, freq_band=(0.04, 0.22))

# Step 1: Memory & Extremums
matrix = ipda.compute_memory_matrix(prices)
sup, res = ipda.extract_extremums(matrix)
spread = res - sup

# Step 2: Fields & Classification
phi = magnet.compute_attraction_vectors(prices, nodes=res)
psi = classifier.classify_regime(prices, volume, spread)

# Step 3: Acceleration & Spectral Gating
g = gamma.compute_feedback(prices)
m = spectral.compute_misalignment(prices, volume)

# Circuit Breaker Logic
safe_gamma = np.where(m['desync_flag'], 0.0, g['gamma_accel'])

In [ ]:
# 📈 Multi-Panel Visualization
fig = plt.figure(figsize=(14, 12))
gs = GridSpec(5, 1, height_ratios=[3, 1, 1, 1, 1], hspace=0.35)

ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1], sharex=ax0)
ax2 = fig.add_subplot(gs[2], sharex=ax0)
ax3 = fig.add_subplot(gs[3], sharex=ax0)
ax4 = fig.add_subplot(gs[4], sharex=ax0)

# 1. Price + IPDA Bands
ax0.plot(timestamps, prices, color='#00ffcc', linewidth=1.2, label='Price')
ax0.plot(timestamps, matrix[:, 0], color='#ff6b6b', alpha=0.6, label='IPDA L20')
ax0.plot(timestamps, matrix[:, 1], color='#ffa500', alpha=0.6, label='IPDA L40')
ax0.fill_between(timestamps, sup, res, color='gray', alpha=0.15, label='Memory Envelope')
ax0.set_title('Explicit Component: IPDA Memory & Time-Decayed References', fontsize=11)
ax0.legend(loc='upper left', framealpha=0.3)
ax0.grid(True, alpha=0.2)

# 2. Potential Attraction
ax1.plot(timestamps, phi, color='#8be9fd', linewidth=1.5)
ax1.axhline(0, color='white', alpha=0.4)
ax1.fill_between(timestamps, 0, phi, where=(phi>0), color='#8be9fd', alpha=0.3)
ax1.set_title('$\\Phi_{\\text{mag}}$ Euclidean Vector Attraction', fontsize=11)
ax1.grid(True, alpha=0.2)

# 3. Gamma Feedback State
colors_gamma = np.where(g['feedback_state']==3, '#ff5555', np.where(g['feedback_state']==2, '#ffb86c', '#50fa7b'))
ax2.bar(timestamps, safe_gamma, color=colors_gamma, alpha=0.85, width=0.6)
ax2.axhline(1.0, color='yellow', linestyle='--', alpha=0.5)
ax2.axhline(-1.0, color='yellow', linestyle='--', alpha=0.5)
ax2.set_title('Gamma Amplifier $\\Gamma$ (Red=Crash Risk, Orange=Overclock)', fontsize=11)
ax2.grid(True, alpha=0.2)

# 4. Spectral Misalignment
ax3.plot(timestamps, m['misalignment'], color='#bd93f9', linewidth=1.5)
ax3.axhline(np.pi/2, color='white', alpha=0.3)
ax3.fill_between(timestamps, 0, m['misalignment'], color='#bd93f9', alpha=0.2)
ax3.set_title('Spectral Misalignment $M$ & Desynchronization Flag', fontsize=11)
ax3.grid(True, alpha=0.2)

# 5. Regime Overlay
reg_colors = {0: '#6272a4', 1: '#ff79c6', 2: '#50fa7b', 3: '#ff5555'}
for code, col in reg_colors.items():
    mask = psi['regime_code'] == code
    ax4.scatter(timestamps[mask], np.ones(mask.sum())*0.5, color=col, alpha=0.7, s=40, label=f'Code {code}')
ax4.set_yticks([])
ax4.set_title('S&D Regime: 0=Neutral 1=Sweep 2=Expansion 3=Exhaustion', fontsize=11)
ax4.legend(loc='upper center', ncol=4, framealpha=0.3)
ax4.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()